# EDA — Municípios do IBGE

Esta análise investiga os registros das **27 capitais**, selecionados na coleta. A base não representa todos os municípios brasileiros.

**Perguntas:**

1. Todas as capitais esperadas estão presentes, com IDs e nomes coerentes?
2. Existem ausências ou duplicações?
3. A UF indicada no arquivo concorda com as hierarquias do IBGE?
4. Como os registros estão distribuídos entre as cinco grandes regiões?

A EDA usa os princípios estrutural, distribucional e relacional da aula. Não consulta APIs nem altera os arquivos da Bronze.


## 1. Bibliotecas e caminhos

Use o kernel do ambiente virtual do projeto. As tabelas são calculadas no Spark; o gráfico usa Matplotlib. Apenas resultados pequenos são levados para o Python.


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


In [ ]:
project_directory = Path.cwd().resolve()

while not (project_directory / "pipeline" / "bronze").is_dir() or not (project_directory / "data").is_dir():

    if project_directory == project_directory.parent:
        raise FileNotFoundError("Execute o notebook a partir de uma pasta dentro do projeto.")

    project_directory = project_directory.parent

MUNICIPALITIES_DIRECTORY = project_directory / "data" / "bronze" / "ibge" / "municipalities"
local_filesystem_jar = project_directory / "pipeline" / "bronze" / "lib" / "hadoop-bare-naked-local-fs-0.1.0.jar"

for required_path in [MUNICIPALITIES_DIRECTORY, local_filesystem_jar]:

    if not required_path.exists():
        raise FileNotFoundError(f"Caminho necessário não encontrado: {required_path}")

print(f"Municípios: {MUNICIPALITIES_DIRECTORY}")


## 2. Sessão Spark

Mantemos a configuração local validada na Bronze, com duas tarefas simultâneas. Execute com um kernel novo para aplicar o classpath desde a criação do Spark.


In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("Viajando com o dado - EDA Municipios IBGE")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.driver.extraClassPath", str(local_filesystem_jar))
    .config("spark.executor.extraClassPath", str(local_filesystem_jar))
    .config("spark.hadoop.fs.file.impl", "com.globalmentor.apache.hadoop.fs.BareLocalFileSystem")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print(f"Versão do Spark: {spark.version}")


## 3. Leitura e estrutura original

A leitura é recursiva porque os arquivos estão em pastas como `SP`. O schema mantém os objetos aninhados recebidos do IBGE.


In [ ]:
municipalities = (
    spark.read
    .option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.parquet")
    .parquet(MUNICIPALITIES_DIRECTORY.resolve().as_posix())
    .cache()
)

record_count = municipalities.count()

print(f"Registros: {record_count}")
print(f"Colunas no nível principal: {len(municipalities.columns)}")

municipalities.printSchema()


## 4. Visão tabular para exploração

Criamos uma visão com os campos internos expostos como colunas simples. Isso facilita a inspeção e evita a aparência de células vazias observada em alguns visualizadores de Parquet. A estrutura original da Bronze permanece intacta.

As hierarquias de microrregiões/mesorregiões e de regiões imediatas/intermediárias representam divisões distintas. Não esperamos que seus IDs sejam iguais. A comparação entre elas será feita nos campos compartilhados de UF e grande região.


In [ ]:
municipalities_view = municipalities.select(
    F.col("id").alias("municipality_id"),
    F.col("nome").alias("municipality_name"),
    F.col("microrregiao.id").alias("microregion_id"),
    F.col("microrregiao.nome").alias("microregion_name"),
    F.col("microrregiao.mesorregiao.id").alias("mesoregion_id"),
    F.col("microrregiao.mesorregiao.nome").alias("mesoregion_name"),
    F.col("microrregiao.mesorregiao.UF.id").alias("legacy_state_id"),
    F.col("microrregiao.mesorregiao.UF.sigla").alias("legacy_state_code"),
    F.col("microrregiao.mesorregiao.UF.nome").alias("legacy_state_name"),
    F.col("microrregiao.mesorregiao.UF.regiao.id").alias("legacy_region_id"),
    F.col("microrregiao.mesorregiao.UF.regiao.sigla").alias("legacy_region_code"),
    F.col("microrregiao.mesorregiao.UF.regiao.nome").alias("legacy_region_name"),
    F.col("`regiao-imediata`.id").alias("immediate_region_id"),
    F.col("`regiao-imediata`.nome").alias("immediate_region_name"),
    F.col("`regiao-imediata`.`regiao-intermediaria`.id").alias("intermediate_region_id"),
    F.col("`regiao-imediata`.`regiao-intermediaria`.nome").alias("intermediate_region_name"),
    F.col("`regiao-imediata`.`regiao-intermediaria`.UF.id").alias("state_id"),
    F.col("`regiao-imediata`.`regiao-intermediaria`.UF.sigla").alias("hierarchy_state_code"),
    F.col("`regiao-imediata`.`regiao-intermediaria`.UF.nome").alias("state_name"),
    F.col("`regiao-imediata`.`regiao-intermediaria`.UF.regiao.id").alias("region_id"),
    F.col("`regiao-imediata`.`regiao-intermediaria`.UF.regiao.sigla").alias("region_code"),
    F.col("`regiao-imediata`.`regiao-intermediaria`.UF.regiao.nome").alias("region_name"),
    "source_file",
    "state_code",
    F.col("microrregiao").isNotNull().alias("has_legacy_hierarchy"),
).cache()

municipalities_view.select(
    "municipality_id", "municipality_name", "state_code", "state_name", "region_code", "region_name"
).orderBy("state_code").show(30, truncate=False)

# Confere explicitamente os campos que apareciam vazios no visualizador.
municipalities_view.filter(F.col("state_code") == "SP").select(
    "municipality_name", "microregion_id", "microregion_name", "mesoregion_name"
).show(truncate=False)


## 5. Dados ausentes

Contamos nulos e textos vazios em cada campo da visão tabular, inclusive os que estavam aninhados. Os percentuais usam o total de registros como denominador.

A ausência de uma hierarquia antiga deve ser investigada com a fonte; não implica automaticamente corrupção ou necessidade de imputação. Os campos obrigatórios para nossas junções são analisados separadamente nas etapas de IDs e cobertura.


In [ ]:
missing_expressions = []

for field in municipalities_view.schema.fields:
    missing_condition = F.col(field.name).isNull()

    if field.dataType.simpleString() == "string":
        missing_condition = missing_condition | (F.trim(F.col(field.name)) == "")

    missing_expressions.append(
        F.sum(F.when(missing_condition, 1).otherwise(0)).alias(field.name)
    )

missing_totals = municipalities_view.agg(*missing_expressions).first()
missing_records = []
missing_cell_count = 0

for column_name in municipalities_view.columns:
    missing_count = int(missing_totals[column_name] or 0)
    missing_percent = 0.0

    if record_count > 0:
        missing_percent = round(100.0 * missing_count / record_count, 2)

    missing_records.append((column_name, missing_count, missing_percent))
    missing_cell_count += missing_count

missing_summary = spark.createDataFrame(
    missing_records,
    "column_name string, missing_count long, missing_percent double",
)

missing_summary.orderBy(F.desc("missing_count"), "column_name").show(30, truncate=False)
missing_legacy_count = municipalities_view.filter(~F.col("has_legacy_hierarchy")).count()

print(f"Registros sem a hierarquia de microrregião: {missing_legacy_count}")


## 6. Duplicações

Buscamos duplicatas exatas e IDs repetidos. No escopo de capitais, também esperamos um único registro por UF.

Nomes de regiões repetidos são esperados: várias capitais pertencem à mesma grande região. Eles não devem ser usados isoladamente como chave de município.


In [ ]:
exact_duplicate_count = record_count - municipalities.dropDuplicates().count()

repeated_ids = (
    municipalities_view.filter(F.col("municipality_id").isNotNull())
    .groupBy("municipality_id")
    .count()
    .filter(F.col("count") > 1)
)

repeated_states = (
    municipalities_view.filter(F.col("state_code").isNotNull())
    .groupBy("state_code")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_id_count = repeated_ids.count()
duplicate_state_count = repeated_states.count()

print(f"Duplicatas exatas adicionais: {exact_duplicate_count}")
print(f"IDs repetidos: {duplicate_id_count}")
print(f"UFs com múltiplos registros: {duplicate_state_count}")

repeated_ids.show(truncate=False)
repeated_states.show(truncate=False)


## 7. Cobertura das capitais

A referência abaixo explicita o escopo configurado na coleta: nome, código IBGE e UF das 27 capitais. Ela não é uma nova consulta à API e deve acompanhar qualquer mudança de escopo.

Comparamos a presença das UFs e a correspondência dos registros com essa referência. Diferenças de espaços nas extremidades e de maiúsculas são ignoradas somente na comparação dos nomes.


In [ ]:
EXPECTED_CAPITALS = [
    ("AC", 1200401, "Rio Branco"),
    ("AL", 2704302, "Maceió"),
    ("AM", 1302603, "Manaus"),
    ("AP", 1600303, "Macapá"),
    ("BA", 2927408, "Salvador"),
    ("CE", 2304400, "Fortaleza"),
    ("DF", 5300108, "Brasília"),
    ("ES", 3205309, "Vitória"),
    ("GO", 5208707, "Goiânia"),
    ("MA", 2111300, "São Luís"),
    ("MG", 3106200, "Belo Horizonte"),
    ("MS", 5002704, "Campo Grande"),
    ("MT", 5103403, "Cuiabá"),
    ("PA", 1501402, "Belém"),
    ("PB", 2507507, "João Pessoa"),
    ("PE", 2611606, "Recife"),
    ("PI", 2211001, "Teresina"),
    ("PR", 4106902, "Curitiba"),
    ("RJ", 3304557, "Rio de Janeiro"),
    ("RN", 2408102, "Natal"),
    ("RO", 1100205, "Porto Velho"),
    ("RR", 1400100, "Boa Vista"),
    ("RS", 4314902, "Porto Alegre"),
    ("SC", 4205407, "Florianópolis"),
    ("SE", 2800308, "Aracaju"),
    ("SP", 3550308, "São Paulo"),
    ("TO", 1721000, "Palmas"),
]

capital_reference = spark.createDataFrame(
    EXPECTED_CAPITALS,
    "state_code string, expected_id long, expected_name string",
)

actual_states = municipalities_view.select("state_code").distinct()
missing_states = capital_reference.select("state_code").join(actual_states, "state_code", "left_anti")
unexpected_states = actual_states.join(capital_reference.select("state_code"), "state_code", "left_anti")

scope_check = municipalities_view.join(capital_reference, "state_code", "left")
name_matches = F.upper(F.trim(F.col("municipality_name"))).eqNullSafe(F.upper(F.trim(F.col("expected_name"))))
id_matches = F.col("municipality_id").eqNullSafe(F.col("expected_id"))
scope_issues = scope_check.filter(F.col("expected_id").isNull() | ~id_matches | ~name_matches)

missing_state_count = missing_states.count()
unexpected_state_count = unexpected_states.count()
scope_issue_count = scope_issues.count()

print(f"Registros encontrados: {record_count}; esperados: {len(EXPECTED_CAPITALS)}")

print(f"UFs esperadas ausentes: {missing_state_count}")
missing_states.show(truncate=False)

print(f"UFs inesperadas: {unexpected_state_count}")
unexpected_states.show(truncate=False)

print(f"Registros que divergem do escopo de capitais: {scope_issue_count}")
scope_issues.select("state_code", "municipality_id", "municipality_name", "expected_id", "expected_name").show(truncate=False)


## 8. IDs e consistência das hierarquias

Verificamos o formato de sete dígitos do código municipal, a correspondência de seu prefixo com o código da UF e a concordância entre a UF do arquivo e a hierarquia regional.

Quando a hierarquia antiga está presente, comparamos os dados de UF e grande região entre as duas estruturas. Não comparamos os IDs das divisões intermediárias com os de mesorregiões: são estruturas diferentes.


In [ ]:
valid_id = F.coalesce(F.col("municipality_id").cast("string").rlike("^[0-9]{7}$"), F.lit(False))
invalid_ids = municipalities_view.filter(~valid_id)

id_prefix_matches = F.substring(F.col("municipality_id").cast("string"), 1, 2).eqNullSafe(F.col("state_id").cast("string"))
file_state_matches = F.col("state_code").eqNullSafe(F.col("hierarchy_state_code"))

hierarchy_check = municipalities_view.withColumn("id_prefix_matches", id_prefix_matches)
hierarchy_check = hierarchy_check.withColumn("file_state_matches", file_state_matches)
hierarchy_issues = hierarchy_check.filter(~F.col("id_prefix_matches") | ~F.col("file_state_matches"))

legacy_comparisons = [
    ("legacy_state_id", "state_id"),
    ("legacy_state_code", "hierarchy_state_code"),
    ("legacy_state_name", "state_name"),
    ("legacy_region_id", "region_id"),
    ("legacy_region_code", "region_code"),
    ("legacy_region_name", "region_name"),
]

legacy_matches = F.lit(True)

for legacy_column, current_column in legacy_comparisons:
    legacy_matches = legacy_matches & F.col(legacy_column).eqNullSafe(F.col(current_column))

legacy_issues = municipalities_view.filter(F.col("has_legacy_hierarchy") & ~legacy_matches)

invalid_id_count = invalid_ids.count()
hierarchy_issue_count = hierarchy_issues.count()
legacy_issue_count = legacy_issues.count()

print(f"IDs ausentes ou fora do formato esperado: {invalid_id_count}")
invalid_ids.select("municipality_id", "municipality_name", "state_code").show(truncate=False)

print(f"Divergências de prefixo ou UF: {hierarchy_issue_count}")
hierarchy_issues.select("municipality_id", "municipality_name", "state_code", "state_id", "hierarchy_state_code").show(truncate=False)

print(f"Divergências entre UF/região das duas hierarquias: {legacy_issue_count}")
legacy_issues.select("municipality_name", "legacy_state_code", "hierarchy_state_code", "legacy_region_code", "region_code").show(truncate=False)


## 9. Princípios distribucional e relacional — grandes regiões

Tratamos IDs como identificadores categóricos: não calculamos média, normalidade ou outliers de códigos. A distribuição relevante aqui é a frequência de capitais por região.

A referência regional registra quantas UFs pertencem a cada grande região. Como selecionamos uma capital por UF, as frequências devem acompanhar essa distribuição. Diferenças entre regiões não representam automaticamente viés de coleta.


In [ ]:
EXPECTED_REGIONS = [
    ("N", 1, "Norte", 7),
    ("NE", 2, "Nordeste", 9),
    ("SE", 3, "Sudeste", 4),
    ("S", 4, "Sul", 3),
    ("CO", 5, "Centro-Oeste", 4),
]

region_reference = spark.createDataFrame(
    EXPECTED_REGIONS,
    "region_code string, expected_region_id long, expected_region_name string, expected_count long",
)

region_check = municipalities_view.join(region_reference, "region_code", "left")
region_id_matches = F.col("region_id").eqNullSafe(F.col("expected_region_id"))
region_name_matches = F.col("region_name").eqNullSafe(F.col("expected_region_name"))
region_issues = region_check.filter(F.col("expected_region_id").isNull() | ~region_id_matches | ~region_name_matches)
region_issue_count = region_issues.count()

region_distribution = (
    municipalities_view.groupBy("region_code", "region_name")
    .count()
    .withColumn("percent", F.round(100.0 * F.col("count") / F.lit(max(record_count, 1)), 2))
)

regional_coverage = region_reference.join(region_distribution, "region_code", "left")
regional_count_issues = regional_coverage.filter(~F.col("count").eqNullSafe(F.col("expected_count")))
regional_count_issue_count = regional_count_issues.count()

region_distribution.orderBy(F.desc("count")).show(truncate=False)
print(f"Registros com código, ID ou nome de região divergente: {region_issue_count}")

region_issues.select("municipality_name", "region_code", "region_id", "region_name").show(truncate=False)
print(f"Regiões com quantidade diferente da esperada: {regional_count_issue_count}")

regional_coverage.select("region_code", "expected_region_name", "expected_count", "count").orderBy("region_code").show(truncate=False)


## 10. Visualização das frequências

O gráfico compara quantidades de capitais, não população, área territorial ou total de municípios. A base cobre o escopo escolhido e não deve ser apresentada como uma amostra de todos os municípios.


In [ ]:
plot_records = region_distribution.filter(F.col("region_code").isNotNull()).orderBy(F.desc("count"), "region_code").collect()
region_labels = []
region_counts = []

for record in plot_records:
    region_labels.append(f"{record['region_name']} ({record['region_code']})")
    region_counts.append(record["count"])

if plot_records:
    figure, axis = plt.subplots(figsize=(9, 5))
    bars = axis.bar(region_labels, region_counts, color="#2d223a")
    axis.bar_label(bars, padding=3)
    axis.set_title("Capitais por grande região — escopo de 27 UFs")
    axis.set_ylabel("Quantidade de capitais")
    axis.set_xlabel("Grande região")
    axis.set_ylim(0, max(region_counts) + 2)
    axis.set_axisbelow(True)
    axis.grid(axis="y", color="lightgray", alpha=0.6)
    figure.tight_layout()
    plt.show()
else:
    print("Não há regiões preenchidas para o gráfico.")


## 11. Síntese e decisões para a Silver

O resumo é calculado a partir das verificações. As contagens podem se sobrepor e as ausências em campos históricos precisam de interpretação, não de preenchimento automático.

**Encaminhamentos:** usar o código municipal como chave de integração; preservar a UF; explicitar os campos aninhados necessários às análises; investigar eventuais divergências antes de corrigir. A frequência regional deve ser interpretada no recorte das capitais.


In [ ]:
checks = [
    ("Diferença em relação aos 27 registros esperados", abs(record_count - len(EXPECTED_CAPITALS))),
    ("Células ausentes na visão tabular", missing_cell_count),
    ("Registros sem a hierarquia antiga", missing_legacy_count),
    ("Duplicatas exatas adicionais", exact_duplicate_count),
    ("IDs repetidos", duplicate_id_count),
    ("UFs com múltiplos registros", duplicate_state_count),
    ("UFs esperadas ausentes", missing_state_count),
    ("UFs inesperadas", unexpected_state_count),
    ("Divergências do escopo de capitais", scope_issue_count),
    ("IDs inválidos", invalid_id_count),
    ("Divergências de prefixo ou UF", hierarchy_issue_count),
    ("Divergências entre as hierarquias", legacy_issue_count),
    ("Divergências de código ou nome de região", region_issue_count),
    ("Regiões com contagem inesperada", regional_count_issue_count),
]

has_issues = False
print(f"Registros analisados: {record_count}")

for check_name, issue_count in checks:
    print(f"{check_name}: {issue_count}")

    if issue_count > 0:
        has_issues = True

if has_issues:
    print("Há pontos para investigar antes de definir os tratamentos da Silver.")
else:
    print("Nenhum problema foi encontrado nas verificações implementadas.")

print("As conclusões se aplicam às capitais selecionadas, não a todos os municípios do Brasil.")


## 12. Encerramento

A visão tabular foi usada apenas em memória. Os arquivos da Bronze não são sobrescritos nem corrigidos por esta análise.


In [ ]:
municipalities_view.unpersist()
municipalities.unpersist()

spark.stop()
